<a href="https://colab.research.google.com/github/StaryDron/PigPostureComputerVision/blob/main/EfficientNet21k_fixed_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ── Instalacja i Kaggle API ───────────────────────────────────────────────────
!pip install timm albumentations -q
from google.colab import files
import os, shutil

print("Wgraj pliki: 'kaggle.json' ORAZ 'corrected_5folds_20260319_190359.csv'")
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
else:
    print("⚠ BRAK kaggle.json!")

!kaggle competitions download -c multi-view-pig-posture-recognition
!unzip -q multi-view-pig-posture-recognition.zip

# ── Importy ───────────────────────────────────────────────────────────────────
import os, ast, copy, re, time
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import torchvision.transforms.functional as TF
import timm

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ── GPU i Stałe ───────────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

CLASS_NAMES = {0: "Lateral_lying_left", 1: "Lateral_lying_right", 2: "Sitting", 3: "Standing", 4: "Sternal_lying"}
NUM_CLASSES = len(CLASS_NAMES)
BASE_DIR    = Path("multiview_pig_posture_recognition")
TRAIN2_IMGS = BASE_DIR / "train2_images"
TEST_IMGS   = BASE_DIR / "test_images"

BATCH_SIZE  = 32
EPOCHS      = 6
FOCAL_GAMMA = 1.5
BBOX_PAD    = 0.00  # Ciasne wycinanie na styk z krawędzią
SAVE_PATH   = "T2_effnetv2_21k_disk.pt"

# ── Przygotowanie Danych i CZYSZCZENIE ETYKIET ────────────────────────────────
train2 = pd.read_csv(BASE_DIR / "train2.csv")

# Aplikacja poprawek (Data Cleaning)
if os.path.exists("corrected_5folds_20260319_190359.csv"):
    print("Wykryto plik z poprawkami! Czyszczenie etykiet...")
    changes_df = pd.read_csv("corrected_5folds_20260319_190359.csv")
    train2.set_index('row_id', inplace=True)
    changes_df.set_index('row_id', inplace=True)
    changes_df['class_id'] = changes_df['corrected_class_id']
    train2.update(changes_df[['class_id']])
    train2.reset_index(inplace=True)
    train2['class_id'] = train2['class_id'].astype(int)
    print(f"Poprawiono {len(changes_df)} błędnych etykiet!")
else:
    print("⚠ UWAGA: Nie wgrano pliku 'corrected_5folds_20260319_190359.csv'! Używam zanieczyszczonych danych.")

test = pd.read_csv(BASE_DIR / "test.csv")

def parse_camera_meta(image_id):
    m = re.match(r"(pen\d+)_(orb|tur)_(cam\d+)_", image_id)
    if m: return m.group(1), m.group(2), m.group(3)
    return "unknown", "unknown", "unknown"

def add_camera_cols(df, source):
    df["source"] = source
    df["bbox_parsed"] = df["bbox"].apply(ast.literal_eval)
    df["pen"]      = df["image_id"].apply(lambda x: parse_camera_meta(x)[0])
    df["cam_type"] = df["image_id"].apply(lambda x: parse_camera_meta(x)[1])
    df["cam_num"]  = df["image_id"].apply(lambda x: parse_camera_meta(x)[2])
    df["camera"]   = df["pen"] + "_" + df["cam_type"] + "_" + df["cam_num"]
    return df

train2 = add_camera_cols(train2, "train2")
test = add_camera_cols(test, "test")

# ── Funkcje pomocnicze ładowania z dysku ──────────────────────────────────────
def load_image(image_id, source):
    folder = {"train2": TRAIN2_IMGS, "test": TEST_IMGS}[source]
    return Image.open(folder / image_id).convert("RGB")

def crop_with_padding(image, bbox, padding=BBOX_PAD, make_square=True):
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1, y1 = x - w * padding, y - h * padding
    x2, y2 = x + w + w * padding, y + h + h * padding
    if make_square:
        side = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, x2, y1, y2 = cx - side/2, cx + side/2, cy - side/2, cy + side/2
    x1, y1 = max(0, int(round(x1))), max(0, int(round(y1)))
    x2, y2 = min(img_w, int(round(x2))), min(img_h, int(round(y2)))
    return image.crop((max(x1, 0), max(y1, 0), max(x2, x1+1), max(y2, y1+1)))

# ── Nasze Stare Funkcje Augmentacji (Bez FDA) ─────────────────────────────────
FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}

def aug_pen2_tur_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, saturation_factor=0.8)
    return TF.adjust_brightness(img, brightness_factor=0.95)

def aug_pen1_tur_cam2(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=1.35)
    return TF.adjust_saturation(img, saturation_factor=0.9)

def aug_pen2_tur_cam2(img):
    img = TF.adjust_brightness(img, brightness_factor=1.3)
    return TF.adjust_saturation(img, saturation_factor=0.85)

def aug_pen2_orb_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=0.6)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    img_np = np.array(img).astype(float)
    img_np[:, :, 0] = (img_np[:, :, 0] * 0.85).clip(0, 255)
    img_np[:, :, 1] = (img_np[:, :, 1] * 1.10).clip(0, 255)
    img_np[:, :, 2] = (img_np[:, :, 2] * 0.80).clip(0, 255)
    img_shifted = Image.fromarray(img_np.clip(0, 255).astype(np.uint8))
    img_grey = TF.to_grayscale(img_shifted, num_output_channels=3)
    blended = (0.65 * img_np + 0.35 * np.array(img_grey).astype(float)).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)

def aug_pen2_orb_cam2(img):
    img = TF.adjust_brightness(img, brightness_factor=0.60)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    img_np, grey_np = np.array(img).astype(float), np.array(TF.to_grayscale(img, num_output_channels=3)).astype(float)
    blended = (0.65 * img_np + 0.35 * grey_np).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)

CAMERA_AUG_FN = {
    "pen2_tur_cam1": (aug_pen2_tur_cam1, True),
    "pen1_tur_cam2": (aug_pen1_tur_cam2, True),
    "pen2_orb_cam1": (aug_pen2_orb_cam1, True),
    "pen2_tur_cam2": (aug_pen2_tur_cam2, False),
    "pen2_orb_cam2": (aug_pen2_orb_cam2, False),
}

# ── Dataset i Dataloader ──────────────────────────────────────────────────────
class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.15): self.std, self.p = std, p
    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = torch.clamp(tensor + torch.randn_like(tensor) * self.std, 0., 1.)
        return tensor

BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(), AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PigDatasetCameraAug(Dataset):
    def __init__(self, df, camera_aug_fn, base_transform, is_train=True):
        self.df = df.reset_index(drop=True)
        self.base_transform = base_transform
        self.camera_aug_fn = camera_aug_fn
        self.samples = []
        for idx, row in self.df.iterrows():
            cam = row.get("camera", "unknown")
            label = int(row["class_id"])
            self.samples.append((idx, False, label))
            if is_train and cam in camera_aug_fn:
                _, flip_label = camera_aug_fn[cam]
                new_label = FLIP_LABEL_MAP[label] if flip_label else label
                self.samples.append((idx, True, new_label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        row_idx, do_aug, label = self.samples[idx]
        row = self.df.iloc[row_idx]

        # ODCZYT Z DYSKU (zamiast RAMu)
        image = load_image(row["image_id"], row["source"])
        crop = crop_with_padding(image, row["bbox_parsed"], padding=BBOX_PAD, make_square=True)

        if do_aug:
            aug_fn, _ = self.camera_aug_fn[row["camera"]]
            crop = aug_fn(crop)

        return self.base_transform(crop), label

train_dataset = PigDatasetCameraAug(train2, CAMERA_AUG_FN, BASE_TRANSFORM, is_train=True)
all_labels = [s[2] for s in train_dataset.samples]

def compute_focal_alpha(labels):
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=NUM_CLASSES)
    inv_f = 1.0 / np.maximum(counts, 1)
    alpha = np.sqrt(inv_f / inv_f.mean())
    return torch.FloatTensor(alpha).to(DEVICE)

class_counts = np.bincount(all_labels, minlength=NUM_CLASSES)
sample_weights = np.array([1.0 / max(class_counts[l], 1) for l in all_labels])
sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

# OPTYMALIZACJA DATALOADERA: prefetch_factor przyspieszy odczyt dysku!
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=2, pin_memory=True, prefetch_factor=2, persistent_workers=True
)
focal_alpha = compute_focal_alpha(all_labels)

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 1.0, alpha: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer('alpha', alpha.float())
        else:
            self.alpha = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal_weight = (1.0 - pt) ** self.gamma
        focal = focal_weight * ce
        return focal.mean()

# ── NOWOŚĆ: Model z Timm (21 milionów zdjęć) ──────────────────────────────────
print("Pobieranie wag modelu z ImageNet-21k (timm)...")
model = timm.create_model(
    'tf_efficientnetv2_s.in21k_ft_in1k',
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=0.40,
    drop_path_rate=0.30
)
model = model.to(DEVICE)

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=focal_alpha)

# NOWOŚĆ: AdamW (Weight Decay)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# NOWOŚĆ: Płynny Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# NOWOŚĆ: AMP (Automatic Mixed Precision)
scaler = torch.cuda.amp.GradScaler()

best_f1 = 0.0
n_batches = len(train_loader)

for epoch in range(EPOCHS):
    model.train()
    all_preds, all_labels_tr = [], []
    running_loss = 0.0

    t0 = time.time()
    print(f"\n--- EPOCH {epoch+1}/{EPOCHS} ---")

    for batch_i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # AMP Włączone
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
        all_labels_tr.extend(labels.cpu().numpy())

        if (batch_i + 1) % 10 == 0 or (batch_i + 1) == n_batches:
            print(f"\r  [Train] Batch {batch_i+1}/{n_batches} | Loss: {loss.item():.4f}", end="", flush=True)

    epoch_time = time.time() - t0
    tr_f1 = f1_score(all_labels_tr, all_preds, average="macro")
    print(f"\n  --> Koniec Epoki ({epoch_time:.1f}s) | Train F1: {tr_f1:.4f} | Avg Loss: {running_loss/n_batches:.4f}")

    scheduler.step()

    if tr_f1 > best_f1:
        best_f1 = tr_f1
        torch.save({"model_state_dict": model.state_dict(), "class_names": CLASS_NAMES}, SAVE_PATH)

print("\nTrening zakończony. Przygotowuję submission...")

# ── Inferencja ────────────────────────────────────────────────────────────────
class PigTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row["image_id"], row["source"]) # ODCZYT Z DYSKU
        crop = crop_with_padding(image, row["bbox_parsed"], padding=BBOX_PAD, make_square=True)
        return self.transform(crop), row["row_id"]

test_loader = DataLoader(PigTestDataset(test, VAL_TRANSFORM), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
model.load_state_dict(torch.load(SAVE_PATH)["model_state_dict"])
model.eval()

all_row_ids, all_preds_test = [], []
n_test_batches = len(test_loader)

with torch.no_grad():
    for batch_i, (images, row_ids) in enumerate(test_loader):
        images = images.to(DEVICE)

        with torch.cuda.amp.autocast():
            preds = model(images).argmax(dim=1).cpu().numpy()

        all_row_ids.extend(row_ids)
        all_preds_test.extend(preds)

        if (batch_i + 1) % 10 == 0 or (batch_i + 1) == n_test_batches:
            print(f"\r  [Test] Batch {batch_i+1}/{n_test_batches}", end="", flush=True)
print()

submission = pd.DataFrame({"row_id": all_row_ids, "class_id": all_preds_test})
sample_sub = pd.read_csv(BASE_DIR / "sample_submission.csv")
submission = submission.set_index("row_id").reindex(sample_sub["row_id"]).reset_index()

submission.to_csv("submission_T2_21k_disk.csv", index=False)
files.download("submission_T2_21k_disk.csv")
files.download(SAVE_PATH)
print("Gotowe! Pliki pobierają się na Twój komputer.")

Wgraj pliki: 'kaggle.json' ORAZ 'corrected_5folds_20260319_190359.csv'


Saving corrected_5folds_20260319_190359.csv to corrected_5folds_20260319_190359 (1).csv
Saving kaggle.json to kaggle.json
multi-view-pig-posture-recognition.zip: Skipping, found more recently modified local copy (use --force to force download)
replace multiview_pig_posture_recognition/pig_posture_classes.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace multiview_pig_posture_recognition/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace multiview_pig_posture_recognition/test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace multiview_pig_posture_recognition/test_images/pen1_tur_cam1_20250920_174649.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace multiview_pig_posture_recognition/test_images/pen1_tur_cam1_20250921_050022.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: Device: cuda
Wykryto plik z poprawkami! Czyszczenie etykiet...
Poprawiono 846 błędnych etykiet!
Pobieranie wag modelu z ImageNet-21k (timm)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]


--- EPOCH 1/6 ---


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gotowe! Pliki pobierają się na Twój komputer.


In [3]:
print("k")

k


In [4]:
# ==============================================================================
# KOMÓRKA 1: FUNKCJE SSL + RUNDA 1
# ==============================================================================
from torch.utils.data import ConcatDataset

# ── 1. Funkcja generująca Pseudo-Labele ───────────────────────────────────────
def generate_pseudo_labels(current_model, test_df, transform):
    print("Generowanie pseudo-etykiet dla zbioru testowego...")
    current_model.eval()

    loader = DataLoader(PigTestDataset(test_df, transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    confs_all, preds_all = [], []
    with torch.no_grad():
        for batch_i, (images, _) in enumerate(loader):
            images = images.to(DEVICE)
            with torch.cuda.amp.autocast():
                outputs = current_model(images)
                probs = F.softmax(outputs, dim=1)
                confs, preds = probs.max(dim=1)

            confs_all.extend(confs.cpu().numpy())
            preds_all.extend(preds.cpu().numpy())

    pseudo_df = test_df.copy()
    pseudo_df['class_id'] = preds_all
    pseudo_df['confidence'] = confs_all
    return pseudo_df

# ── 2. Dataset dla Pseudo-Labeli ──────────────────────────────────────────────
SSL_TRANSFORM = transforms.Compose([
    transforms.RandomRotation(degrees=90), # Obrót o 90 stopni żeby model nie zapamiętał pseudo-labeli "na blachę"
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PseudoDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row['image_id'], row['source']) # Odczyt z dysku
        crop = crop_with_padding(image, row['bbox_parsed'], padding=BBOX_PAD, make_square=True)
        return self.transform(crop), int(row['class_id'])

# ── 3. Główna pętla jednej rundy SSL ──────────────────────────────────────────
def run_ssl_round(current_model, pseudo_df, threshold, lr, epochs, round_num):
    print(f"\n{'='*50}\nSSL RUNDA {round_num} | Próg pewności: {threshold}\n{'='*50}")

    pseudo_high = pseudo_df[pseudo_df['confidence'] >= threshold]
    print(f"Wykorzystuję {len(pseudo_high)} z {len(pseudo_df)} zdjęć testowych.")

    pseudo_ds = PseudoDataset(pseudo_high, SSL_TRANSFORM)
    combined_ds = ConcatDataset([train_dataset, pseudo_ds])

    all_labels_combined = all_labels + pseudo_high['class_id'].tolist()

    ssl_alpha = compute_focal_alpha(all_labels_combined)
    criterion_ssl = FocalLoss(gamma=FOCAL_GAMMA, alpha=ssl_alpha)

    class_counts_comb = np.bincount(all_labels_combined, minlength=NUM_CLASSES)
    sample_weights_comb = np.array([1.0 / max(class_counts_comb[l], 1) for l in all_labels_combined])
    sampler_comb = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights_comb), num_samples=len(sample_weights_comb), replacement=True)

    loader_ssl = DataLoader(combined_ds, batch_size=BATCH_SIZE, sampler=sampler_comb, num_workers=2, pin_memory=True, prefetch_factor=2, persistent_workers=True)

    optimizer_ssl = torch.optim.AdamW(current_model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler_ssl = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ssl, T_max=epochs, eta_min=1e-6)

    best_f1_ssl = 0.0
    best_state_ssl = None

    for epoch in range(epochs):
        current_model.train()
        all_preds, all_labels_tr = [], []
        running_loss = 0.0
        t0 = time.time()

        for images, labels in loader_ssl:
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer_ssl.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast():
                outputs = current_model(images)
                loss = criterion_ssl(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_ssl)
            torch.nn.utils.clip_grad_norm_(current_model.parameters(), 1.0)
            scaler.step(optimizer_ssl)
            scaler.update()

            running_loss += loss.item()
            all_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
            all_labels_tr.extend(labels.cpu().numpy())

        tr_f1 = f1_score(all_labels_tr, all_preds, average="macro")
        print(f"  Epoka {epoch+1}/{epochs} ({(time.time()-t0):.1f}s) | Train F1: {tr_f1:.4f} | Loss: {running_loss/len(loader_ssl):.4f}")

        scheduler_ssl.step()

        if tr_f1 > best_f1_ssl:
            best_f1_ssl = tr_f1
            best_state_ssl = copy.deepcopy(current_model.state_dict())

    current_model.load_state_dict(best_state_ssl)
    return current_model

# ── 4. WYKONANIE RUNDY 1 (Próg 0.95, 2 epoki) ─────────────────────────────────
pseudo_all_r1 = generate_pseudo_labels(model, test, VAL_TRANSFORM)
model = run_ssl_round(model, pseudo_all_r1, threshold=0.95, lr=2e-5, epochs=2, round_num=1)

# ── 5. ZAPIS I INFERENCJA PO RUNDZIE 1 ────────────────────────────────────────
SAVE_SSL_R1_PATH = "T2_effnetv2_21k_SSL_R1.pt"
torch.save({"model_state_dict": model.state_dict(), "class_names": CLASS_NAMES}, SAVE_SSL_R1_PATH)

print("\nPrzygotowuję submisję po RUNDZIE 1...")
test_loader = DataLoader(PigTestDataset(test, VAL_TRANSFORM), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
model.eval()

all_row_ids, all_preds_test = [], []
with torch.no_grad():
    for images, row_ids in test_loader:
        images = images.to(DEVICE)
        with torch.cuda.amp.autocast():
            preds = model(images).argmax(dim=1).cpu().numpy()
        all_row_ids.extend(row_ids)
        all_preds_test.extend(preds)

submission = pd.DataFrame({"row_id": all_row_ids, "class_id": all_preds_test})
sample_sub = pd.read_csv(BASE_DIR / "sample_submission.csv")
submission = submission.set_index("row_id").reindex(sample_sub["row_id"]).reset_index()

submission.to_csv("submission_T2_21k_SSL_R1.csv", index=False)
from google.colab import files
files.download("submission_T2_21k_SSL_R1.csv")
files.download(SAVE_SSL_R1_PATH)
print("Gotowe! Pliki z Rundy 1 pobierają się na dysk.")

Generowanie pseudo-etykiet dla zbioru testowego...


/tmp/ipykernel_1609/2188145605.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



SSL RUNDA 1 | Próg pewności: 0.95
Wykorzystuję 8044 z 11708 zdjęć testowych.


/tmp/ipykernel_1609/2188145605.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Epoka 1/2 (768.6s) | Train F1: 0.9631 | Loss: 0.0495


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gotowe! Pliki z Rundy 1 pobierają się na dysk.


In [ ]:
# ==============================================================================
# KOMÓRKA 2: SSL RUNDA 2 + OSTATECZNY ZAPIS
# ==============================================================================

# ── 1. WYKONANIE RUNDY 2 (Próg 0.90, 2 epoki) ─────────────────────────────────
# Generujemy pseudo-labele na nowo, bo model po Rundzie 1 jest już mądrzejszy!
pseudo_all_r2 = generate_pseudo_labels(model, test, VAL_TRANSFORM)
model = run_ssl_round(model, pseudo_all_r2, threshold=0.90, lr=1e-5, epochs=2, round_num=2)

# ── 2. ZAPIS I INFERENCJA PO RUNDZIE 2 ────────────────────────────────────────
SAVE_SSL_R2_PATH = "T2_effnetv2_21k_SSL_R2.pt"
torch.save({"model_state_dict": model.state_dict(), "class_names": CLASS_NAMES}, SAVE_SSL_R2_PATH)

print("\nPrzygotowuję OSTATECZNĄ submisję po RUNDZIE 2...")
test_loader = DataLoader(PigTestDataset(test, VAL_TRANSFORM), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
model.eval()

all_row_ids, all_preds_test = [], []
with torch.no_grad():
    for images, row_ids in test_loader:
        images = images.to(DEVICE)
        with torch.cuda.amp.autocast():
            preds = model(images).argmax(dim=1).cpu().numpy()
        all_row_ids.extend(row_ids)
        all_preds_test.extend(preds)

submission = pd.DataFrame({"row_id": all_row_ids, "class_id": all_preds_test})
sample_sub = pd.read_csv(BASE_DIR / "sample_submission.csv")
submission = submission.set_index("row_id").reindex(sample_sub["row_id"]).reset_index()

submission.to_csv("submission_T2_21k_SSL_R2.csv", index=False)
from google.colab import files
files.download("submission_T2_21k_SSL_R2.csv")
files.download(SAVE_SSL_R2_PATH)
print("Gotowe! Ostateczne pliki z Rundy 2 pobierają się na dysk.")

Generowanie pseudo-etykiet dla zbioru testowego...


/tmp/ipykernel_1609/2188145605.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



SSL RUNDA 2 | Próg pewności: 0.9
Wykorzystuję 9083 z 11708 zdjęć testowych.


/tmp/ipykernel_1609/2188145605.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
